<a href="https://colab.research.google.com/github/ujju2020/ujju2020/blob/main/Gemini_based_storytelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini-Based Interactive Storytelling Adventure
## Essentials of Generative AI, Prompt Engineering 

**Objective:** Unleash creativity by using Google Gemini as a collaborative tool to build an interactive story with unique and engaging narratives — no additional platforms required.

---

### Project Flow
1. Define the story theme
2. Introduce main characters
3. Begin the story (set the scene)
4. Participant input & AI-generated responses
5. Decision-making and branching paths
6. Iteration and refinement
7. Conclusion
---

## Step 0: Setup — Install & Import Libraries

**Get your free Gemini API key:** [https://aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)

In [ ]:
# Install the Google Generative AI SDK
!pip install google-generativeai -q

In [ ]:
import google.generativeai as genai
import textwrap
import os

# ---------------------------------------------------------------------------
# IMPORTANT: Replace with your actual Gemini API key
# Get one free at: https://aistudio.google.com/app/apikey
# Best practice: use an environment variable
#   set GEMINI_API_KEY=AIza...
# ---------------------------------------------------------------------------
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR-API-KEY")
genai.configure(api_key=GEMINI_API_KEY)

print("Google Generative AI library loaded. API key status:",
      "Set via env var" if os.getenv("GEMINI_API_KEY") else "Using placeholder — replace YOUR_GEMINI_API_KEY_HERE")

Google Generative AI library loaded. API key status: Using placeholder — replace YOUR_GEMINI_API_KEY_HERE


In [ ]:
print(f"Current GEMINI_API_KEY value: {GEMINI_API_KEY}")

Current GEMINI_API_KEY value: YOUR-API-KEY


In [ ]:
print("\n--- Testing API Key Configuration ---")
try:
    # Attempt to list models to verify API key is active
    for m in genai.list_models():
        if "gemini" in m.name:
            print(f"Found Gemini model: {m.name}")
            break # Print one and exit, just to confirm connectivity
    else:
        print("No Gemini models found, but API call succeeded. Key might be valid.")
    print("API key configuration test: SUCCESS!\n")
except Exception as e:
    print(f"API key configuration test: FAILED! Error: {e}\n")
    print("Please ensure your API key in cell 'dd3262fd' is correct and not a placeholder.")



--- Testing API Key Configuration ---
Found Gemini model: models/gemini-2.5-flash
API key configuration test: SUCCESS!



## Step 1: Define the Story Theme

Choose a genre or theme for your adventure. Supported examples:
- `fantasy` — swords, magic, dragons
- `science fiction` — space exploration, AI, interstellar travel
- `mystery` — detective noir, hidden clues, whodunits
- `adventure` — exploration, treasure hunts, survival

Modify the variables below to customise your story.

In [ ]:
# ── Customise your story here ──────────────────────────────────────────────
STORY_THEME   = "fantasy"          # genre / theme
STORY_SETTING = "an ancient kingdom hidden beneath a mountain"
STORY_TONE    = "mysterious and adventurous"
# ──────────────────────────────────────────────────────────────────────────

print(f"Theme    : {STORY_THEME}")
print(f"Setting  : {STORY_SETTING}")
print(f"Tone     : {STORY_TONE}")

Theme    : fantasy
Setting  : an ancient kingdom hidden beneath a mountain
Tone     : mysterious and adventurous


## Step 2: Introduce the Main Characters

In [ ]:
# Define your characters as a list of dictionaries
characters = [
    {
        "name"        : "Elara",
        "role"        : "Protagonist — young cartographer",
        "personality" : "Curious, brave, and fiercely independent",
        "background"  : "Raised by scholars in the Citadel of Maps; obsessed with discovering uncharted lands"
    },
    {
        "name"        : "Kael",
        "role"        : "Ally — exiled knight",
        "personality" : "Stoic, loyal, haunted by past failures",
        "background"  : "Once the king's champion; banished after a battle he refused to fight"
    },
    {
        "name"        : "Seraphon",
        "role"        : "Antagonist — shadow sorcerer",
        "personality" : "Cunning, ruthless, believes the ends justify the means",
        "background"  : "Seeks the ancient power buried beneath the mountain to reshape the world"
    }
]

# Display character profiles
print("=" * 55)
print("           CAST OF CHARACTERS")
print("=" * 55)
for char in characters:
    print(f"\nName        : {char['name']}")
    print(f"Role        : {char['role']}")
    print(f"Personality : {char['personality']}")
    print(f"Background  : {char['background']}")
    print("-" * 55)

           CAST OF CHARACTERS

Name        : Elara
Role        : Protagonist — young cartographer
Personality : Curious, brave, and fiercely independent
Background  : Raised by scholars in the Citadel of Maps; obsessed with discovering uncharted lands
-------------------------------------------------------

Name        : Kael
Role        : Ally — exiled knight
Personality : Stoic, loyal, haunted by past failures
Background  : Once the king's champion; banished after a battle he refused to fight
-------------------------------------------------------

Name        : Seraphon
Role        : Antagonist — shadow sorcerer
Personality : Cunning, ruthless, believes the ends justify the means
Background  : Seeks the ancient power buried beneath the mountain to reshape the world
-------------------------------------------------------


## Step 3: Build the System Instruction (Story Engine)

In Gemini, the storytelling persona is set via the `system_instruction` parameter when creating the model — equivalent to the system prompt in ChatGPT. This tells Gemini the rules of the world, the characters, the tone, and its role as a collaborative narrator.

> **Gemini vs ChatGPT:** ChatGPT receives the system prompt as a `{role: "system"}` message in every API call. Gemini accepts it once at model creation time via `system_instruction`, keeping it cleanly separated from the conversation history.

In [ ]:
def build_character_block(chars):
    """Convert characters list into a structured text block for the system instruction."""
    lines = []
    for c in chars:
        lines.append(
            f"- {c['name']} ({c['role']}): {c['personality']}. Background: {c['background']}"
        )
    return "\n".join(lines)


SYSTEM_INSTRUCTION = f"""You are a master storyteller and collaborative narrator for an interactive {STORY_THEME} adventure.

Setting: {STORY_SETTING}.
Tone: {STORY_TONE}.

Characters:
{build_character_block(characters)}

Your responsibilities:
1. Continue the story based on the participant's input.
2. Keep each response vivid, immersive, and 2-4 paragraphs long.
3. End each response with exactly TWO numbered choices for the participant to pick from, e.g.:
   Choice 1: [action A]
   Choice 2: [action B]
4. Maintain internal consistency with previous events.
5. Naturally weave in all main characters where appropriate.
6. When the participant types 'end story', wrap up the narrative with a satisfying conclusion."""

print("System instruction built successfully.")
print(textwrap.fill(SYSTEM_INSTRUCTION[:200] + "...", width=80))

System instruction built successfully.
You are a master storyteller and collaborative narrator for an interactive
fantasy adventure.  Setting: an ancient kingdom hidden beneath a mountain. Tone:
mysterious and adventurous.  Characters: - E...


## Step 4: Begin the Story — Set the Scene

We initialise a Gemini `GenerativeModel` with the story engine system instruction, then start a `ChatSession`. The session automatically retains the full conversation history — no manual list management needed.

In [ ]:
def create_story_model(system_instruction=SYSTEM_INSTRUCTION):
    """Create a Gemini GenerativeModel configured as the story narrator."""
    return genai.GenerativeModel(
        model_name="gemini-2.5-flash",   # swap to "gemini-1.5-pro" for richer responses
        system_instruction=system_instruction,
        generation_config=genai.GenerationConfig(
            temperature=0.85,
            max_output_tokens=600,
        ),
    )

def chat_with_gemini(user_input, chat_session):
    """
    Send a message to the Gemini story narrator and return the reply.
    The ChatSession automatically appends both the user message and
    the model response to its internal history.

    Parameters
    ----------
    user_input   : str               — participant's action or choice
    chat_session : genai.ChatSession — active session with story history

    Returns
    -------
    str — the narrator's response text
    """
    response = chat_session.send_message(user_input)
    return response.text


# ── Initialise the story ────────────────────────────────────────────────────
story_model   = create_story_model()
chat_session  = story_model.start_chat(history=[])   # fresh story session

opening_prompt = (
    "Begin the story. Set the scene vividly, introduce Elara, hint at the danger ahead, "
    "and end with two choices for the participant."
)

opening_response = chat_with_gemini(opening_prompt, chat_session)

print("=" * 60)
print("                  THE STORY BEGINS")
print("=" * 60)
print()
print(textwrap.fill(opening_response, width=80))

                  THE STORY BEGINS

The air within the Citadel of Maps was a living thing, thick with the scent of
aged parchment, lamp


## Step 5: Participant Input & AI-Generated Responses

Run the cell below to make a choice and advance the story.
Change the `participant_input` variable to whatever you want to do next.

In [ ]:
# ── Enter your choice or action here ───────────────────────────────────────
participant_input = "Choice 1"   # Change to Choice 2, or describe your own action
# ───────────────────────────────────────────────────────────────────────────

ai_response = chat_with_gemini(participant_input, chat_session)

print(f">> You chose: {participant_input}")
print()
print(textwrap.fill(ai_response, width=80))

>> You chose: Choice 1

The air within the Citadel of Maps was a living thing, thick with the scent of
aged parchment, lamp oil, and the faint, metallic tang of ink. Dust motes danced
in the golden cones of light cast by the flickering oil lamps, illuminating
shelves stacked to the vaulted ceilings with scrolls and tomes. Elara, her dark
hair often escaping its braid to frame a smudged cheek, was lost to the world,
hunched over a brittle, oversized map spread across her workbench. Her brow was
furrowed in concentration, a magnifying glass held steadily in one


## Step 6: Decision-Making — Branching Paths

Continue the adventure cell by cell. Each subsequent cell sends a new input to Gemini and the `ChatSession` keeps the full history so the story stays consistent.

> **Tip:** You can describe custom actions instead of just picking Choice 1 or 2, e.g. `"Elara decides to sneak past the guards using her cloak"` — the model will adapt.

In [ ]:
# ── Advance the story (turn 2) ──────────────────────────────────────────────
participant_input = "Choice 2"   # Change to your preferred action
# ───────────────────────────────────────────────────────────────────────────

ai_response = chat_with_gemini(participant_input, chat_session)

print(f">> You chose: {participant_input}")
print()
print(textwrap.fill(ai_response, width=80))

>> You chose: Choice 2

The air within the Citadel of Maps was a living thing, thick with the scent of
aged parchment, lamp oil, and the faint, metallic tang of ink. Dust motes danced
in the golden cones of light cast by the flickering oil lamps, illuminating
shelves stacked to the vaulted ceilings with scrolls and tomes. Elara, her dark
hair often escaping its braid to frame a smudged cheek, was lost to the world,
hunched over a brittle, oversized map spread across her workbench. Her brow was
furrowed in concentration, a magnifying glass held steadily in one hand, tracing
the faded lines of an ancient, incomplete chart. It depicted the sprawling,
labyrinthine tunnels beneath the mountain kingdom, a network of forgotten
passages and rumored chambers that had always called to her adventurous spirit.
Her current obsession was a particular section marked only by a faded symbol – a
stylized shadow devouring a sun – and a cryptic note: *“The Heart of Stone,
where echoes awaken.”* The schola

In [ ]:
# ── Advance the story (turn 3) ──────────────────────────────────────────────
participant_input = "Elara convinces Kael to help her decode the ancient inscription on the door"
# ───────────────────────────────────────────────────────────────────────────

ai_response = chat_with_gemini(participant_input, chat_session)

print(f">> Your action: {participant_input}")
print()
print(textwrap.fill(ai_response, width=80))

>> Your action: Elara convinces Kael to help her decode the ancient inscription on the door

Elara, her mind alight with purpose, had indeed chosen to secretly prepare for
an expedition. For days, she moved


## Step 7: Iteration & Refinement

Use the helpers below to review the full story so far, refine past inputs, or regenerate a response with a tweaked prompt.

> **Gemini note:** `chat_session.history` is a list of `Content` objects (each with `.role` and `.parts`). The helpers below read from this native history — no separate list to maintain.

In [ ]:
def print_full_story(session):
    """Print the complete story conversation from the Gemini ChatSession history."""
    print("=" * 60)
    print("              FULL STORY SO FAR")
    print("=" * 60)
    turn = 1
    for i, content in enumerate(session.history):
        role = content.role          # "user" or "model"
        text = content.parts[0].text
        if role == "user":
            print(f"\n[YOU — Turn {turn}]")
            print(textwrap.fill(text, width=80))
        else:
            print(f"\n[NARRATOR — Turn {turn}]")
            print(textwrap.fill(text, width=80))
            turn += 1
        print("-" * 60)


print_full_story(chat_session)

              FULL STORY SO FAR

[YOU — Turn 1]
Begin the story. Set the scene vividly, introduce Elara, hint at the danger
ahead, and end with two choices for the participant.
------------------------------------------------------------

[NARRATOR — Turn 1]
The air within the Citadel of Maps was a living thing, thick with the scent of
aged parchment, lamp
------------------------------------------------------------

[YOU — Turn 2]
Choice 1
------------------------------------------------------------

[NARRATOR — Turn 2]
The air within the Citadel of Maps was a living thing, thick with the scent of
aged parchment, lamp oil, and the faint, metallic tang of ink. Dust motes danced
in the golden cones of light cast by the flickering oil lamps, illuminating
shelves stacked to the vaulted ceilings with scrolls and tomes. Elara, her dark
hair often escaping its braid to frame a smudged cheek, was lost to the world,
hunched over a brittle, oversized map spread across her workbench. Her brow wa

In [ ]:
def regenerate_last_response(session):
    """
    Remove the last model turn from the history and regenerate it.
    Useful for getting a different version of the same story beat.

    Note: Gemini ChatSession history is a regular list — we can pop the last
    entry and re-send the last user message to get a fresh response.
    """
    history = session.history
    if len(history) >= 2 and history[-1].role == "model":
        # Remove the last model response
        history.pop()
        # The last item is now the user message — re-send it
        last_user_text = history[-1].parts[0].text
        history.pop()   # also remove the user turn so send_message re-adds it cleanly
        new_response = session.send_message(last_user_text)
        print("[Regenerated Response]")
        print()
        print(textwrap.fill(new_response.text, width=80))
    else:
        print("Nothing to regenerate.")


# Uncomment the line below to regenerate the last AI response
# regenerate_last_response(chat_session)

## Step 8: Conclusion — End the Story

When you're ready to conclude the adventure, run the cell below. The narrator will craft a satisfying ending based on everything that happened.

In [ ]:
# Send the special keyword to trigger a conclusion
conclusion_input    = "end story"
conclusion_response = chat_with_gemini(conclusion_input, chat_session)

print("=" * 60)
print("                THE CONCLUSION")
print("=" * 60)
print()
print(textwrap.fill(conclusion_response, width=80))

                THE CONCLUSION

Elara, her mind alight with purpose, had indeed chosen to secretly prepare for
an expedition. For days, she moved


## Step 9: Showcase & Share — Export the Story

In [ ]:
def export_story_to_txt(session, filename="my_gemini_story.txt"):
    """
    Export the full story to a plain-text file for sharing.
    Only the narrator (model) turns are written — producing a clean narrative.
    """
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"INTERACTIVE STORY — {STORY_THEME.upper()} ADVENTURE (Powered by Google Gemini)\n")
        f.write(f"Setting : {STORY_SETTING}\n")
        f.write("=" * 60 + "\n\n")
        chapter = 1
        for content in session.history:
            if content.role == "model":
                f.write(f"--- Chapter {chapter} ---\n")
                f.write(content.parts[0].text + "\n\n")
                chapter += 1
    print(f"Story exported to '{filename}'")


export_story_to_txt(chat_session, "my_gemini_story.txt")

Story exported to 'my_gemini_story.txt'


In [ ]:
# Optional: Display the exported story content inline
with open("my_gemini_story.txt", "r", encoding="utf-8") as f:
    print(f.read())

INTERACTIVE STORY — FANTASY ADVENTURE (Powered by Google Gemini)
Setting : an ancient kingdom hidden beneath a mountain

--- Chapter 1 ---
The air within the Citadel of Maps was a living thing, thick with the scent of aged parchment, lamp

--- Chapter 2 ---
The air within the Citadel of Maps was a living thing, thick with the scent of aged parchment, lamp oil, and the faint, metallic tang of ink. Dust motes danced in the golden cones of light cast by the flickering oil lamps, illuminating shelves stacked to the vaulted ceilings with scrolls and tomes. Elara, her dark hair often escaping its braid to frame a smudged cheek, was lost to the world, hunched over a brittle, oversized map spread across her workbench. Her brow was furrowed in concentration, a magnifying glass held steadily in one

--- Chapter 3 ---
The air within the Citadel of Maps was a living thing, thick with the scent of aged parchment, lamp oil, and the faint, metallic tang of ink. Dust motes danced in the golden cones o

---
## Summary

| Step | Task | Status |
|------|------|--------|
| 1 | Define story theme (`fantasy`) | ✅ |
| 2 | Introduce characters (Elara, Kael, Seraphon) | ✅ |
| 3 | Build system instruction / story engine | ✅ |
| 4 | Begin the story — set the scene | ✅ |
| 5 | Participant input & Gemini responses | ✅ |
| 6 | Branching paths via choices | ✅ |
| 7 | Iteration & refinement helpers | ✅ |
| 8 | Conclude the story | ✅ |
| 9 | Export & share the story | ✅ |

**To try a different genre:** Change `STORY_THEME`, `STORY_SETTING`, `STORY_TONE`, and the `characters` list in Steps 1–2, then re-run all cells from Step 3 onward.

---
### ChatGPT vs Gemini — Key Differences in This Notebook

| Aspect | ChatGPT (OpenAI) | Google Gemini |
|--------|----------------|---------------|
| Library | `openai` | `google-generativeai` |
| System prompt | `{role: "system"}` in messages list | `system_instruction` on the model |
| Conversation history | Manual list `[{role, content}]` | Native `ChatSession` — automatic |
| API call | `openai.chat.completions.create(...)` | `chat_session.send_message(...)` |
| History access | Your own list variable | `chat_session.history` |
| Free tier model | `gpt-3.5-turbo` (limited) | `gemini-1.5-flash` (generous free tier) |